# [Trend] Autonomous Forecasters: The `LLMAgentForecaster`

### Professional sktime-native Agentic Forecasting with Semantic Intelligence

This notebook demonstrates the **`LLMAgentForecaster`** (v2.0). We've added a **Transparency & Diagnostics** layer that allows the agent to not just forecast, but to act as a data auditor.

**Strategic Features:**
1. **Registry Intelligence**: Dynamic scanning of all installed sktime estimators.
2. **Semantic Anomaly Detection**: Cross-references textual logs with numerical trends to flag data integrity issues.
3. **Autonomous Benchmarking**: Calculates 'Accuracy Lift' against naive baselines automatically.
4. **Rich Reporting**: Generates high-impact analytics reports for business stakeholders.

In [ ]:
import pandas as pd
import numpy as np
import os
from IPython.display import Markdown, display
from llm_pipeline_forecaster import LLMAgentForecaster, plot_forecast, generate_summary_report

api_key = os.environ.get("GROQ_API_KEY", "")

## 1. Scenario: Catching a Recording Error
We'll simulate a situation where the text logs say the store was **closed**, but the sales numbers remain high. A blind model would just fit the data; our Agent will flag the inconsistency.

In [ ]:
dates = pd.date_range(start="2024-01-01", periods=30, freq="D")
y = pd.Series(100 + np.random.normal(0, 5, 30), index=dates)
X = pd.DataFrame({"log": ["Normal ops"] * 30}, index=dates)

# THE ANOMALY: Log says closed, but sales are fine (y.iloc[10] is not dropped)
X.iloc[10] = "CRITICAL: Store closed for emergency plumbing"
print("Injecting intentional data recording error at index 10...")

agent = LLMAgentForecaster(
    prompt="Analyze this data and forecast next 5 days.",
    api_key=api_key,
    text_column="log",
    text_schema={"risk": "float"}
)

agent.fit(y, X=X)

display(Markdown(generate_summary_report(agent)))

## 2. Visualization
Observe the chosen model and the forecast.

In [ ]:
y_pred = agent.predict(fh=[1,2,3,4,5])
plot_forecast(agent, y, y_pred)